# V46 · Healthy purchasing over the household life cycle

This notebook estimates how households allocate spending within five food categories and how that allocation changes across latent household-year states. It compares a two-state hidden Markov model (HMM) with a two-class model (LCM) in which each household keeps the same class throughout the panel.

**Running the analysis.** Set the Parquet path and run mode in the next cell, then restart the kernel and choose **Run All**. The notebook uses Python, NumPy, pandas, SciPy and DuckDB. The optional `threadpoolctl` package also controls BLAS threads in an already-initialized Python session.

| Run mode | Data and work performed | Destination |
|---|---|---|
| `full_estimation` | Research panel, both models, 32 starts per model and 500 paired household bootstrap draws by default | `OUTPUT_DIR` |
| `validate_only` | Research panel, input checks, audits and annual sufficient statistics | `OUTPUT_DIR` |
| `synthetic` | Generated panel of 240 households, three starts per model and three bootstrap draws for a reproducibility check | `V46_synthetic_results` in the working directory |

## 1. Run settings

The default input path and results folder match the original V46 run. On another machine, edit `INPUT_PATH`. To resume an interrupted fit, rerun with the same data, specification and output folder. The checkpoint directory holds completed full-sample fits and individual bootstrap draws. A first run of this revision creates its own checkpoint directory.

In [ ]:
import os
from pathlib import Path

# Edit these paths when running on another machine.
INPUT_PATH = Path(
    r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel_sugar.parquet"
)
OUTPUT_DIR = INPUT_PATH.parent / "V46_Conditional_Composition_Results"
RUN_MODE = "full_estimation"  # full_estimation | validate_only | synthetic

# Replication settings used for the reported models.
SEED = 20261118
N_STARTS = 32
BOOTSTRAP_REPS = 500
CHECK_DUPLICATE_TRIPS = True
MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 1
YEAR_GAP_POLICY = "observed_step"
SPENDING_NEGATIVE_TOL = 1e-8

# CPU budget for data preparation. Independent fits share this budget.
HMM_NUM_THREADS = int(os.getenv("HMM_NUM_THREADS", min(4, os.cpu_count() or 1)))
# Two fits share the annual panel by default. Use 1 on a memory-limited machine.
FIT_WORKERS = int(os.getenv("HMM_FIT_WORKERS", min(2, HMM_NUM_THREADS)))
DUCKDB_MEMORY_LIMIT = "4GB"
if not 1 <= FIT_WORKERS <= HMM_NUM_THREADS:
    raise ValueError("Require 1 <= HMM_FIT_WORKERS <= HMM_NUM_THREADS.")
BLAS_THREADS = HMM_NUM_THREADS if FIT_WORKERS == 1 else 1
for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[variable] = str(BLAS_THREADS)

## 2. Category definitions and covariates

The first component of each category is the numerator of the reported healthy spending share. In the exported tables, “healthy” is shorthand for the preferred component under these study definitions.

| Category | Preferred component | Other component |
|---|---|---|
| Fruit and vegetables | Fresh | Nonfresh |
| Yogurt | Regular sugar | High sugar |
| Breakfast cereal | Low and medium sugar | High sugar |
| Snacks | Healthy snack group | Unhealthy snack group |
| Beverages | Healthier beverage group and all three milk spending columns | Less healthy beverage group |

The pure-basket equation includes the child indicator `K_1`, income in $10,000 units, family size and age in decades. The transition equations use the six life events observed in the destination year. The code below maps these definitions to source columns and output labels.

In [ ]:
from tempfile import TemporaryDirectory
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import nullcontext
from functools import partial
import hashlib
import inspect
import json

import duckdb
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize
from scipy.special import betaln, digamma, expit, logit
from scipy.stats import norm

try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

TRANSITION_EVENT_COLUMNS = [
    "event_new_child",
    "getting_married",
    "getting_divorce_or_separation",
    "moving",
    "getting_job",
    "retirement",
]
TRANSITION_X_COLS = ["intercept", *TRANSITION_EVENT_COLUMNS]
GATE_COVARIATE_COLS = ["K_1", "income10k_imputed", "family_size_numeric", "age"]

# Labels and source expenditure columns are in preferred / other order.
CATEGORY_GROUPS = {
    "fv": (("fresh", ["fresh_spending"]), ("nonfresh", ["non_fresh_spending"])),
    "yogurt": (
        ("regular_sugar", ["regular_sugar_spending"]),
        ("high_sugar", ["high_sugar_spending"]),
    ),
    "cereal": (
        (
            "healthy_low_medium",
            ["low_sugar_cereal_spending", "medium_sugar_cereal_spending"],
        ),
        ("high_sugar", ["high_sugar_cereal_spending"]),
    ),
    "snack": (
        ("healthy", ["healthy_snack_spending"]),
        ("unhealthy", ["unhealthy_snack_spending"]),
    ),
    "beverage": (
        (
            "healthier",
            [
                "healthy_beverage_spending",
                "skimmed_milk_spending",
                "low_fat_milk_spending",
                "other_milk_spending",
            ],
        ),
        ("less_healthy", ["unhealthy_beverage_spending"]),
    ),
}
CATEGORY_NAMES = list(CATEGORY_GROUPS)
SHARE_COMPONENTS = {
    c: [cols for _, cols in groups] for c, groups in CATEGORY_GROUPS.items()
}
CATEGORY_CONFIG = {}
for category, groups in CATEGORY_GROUPS.items():
    zero_label = "no_classified_purchase" if category == "cereal" else "no_purchase"
    CATEGORY_CONFIG[category] = dict(
        outcomes=dict(enumerate([zero_label, *[label for label, _ in groups]])),
        spending=[col for _, cols in groups for col in cols],
    )
# Preserve the legacy order of addition for the reported beverage budget.
CATEGORY_CONFIG["beverage"]["spending"] = [
    "healthy_beverage_spending",
    "unhealthy_beverage_spending",
    "skimmed_milk_spending",
    "low_fat_milk_spending",
    "other_milk_spending",
]

## 3. From trip records to household histories

The input is the 2008–2017 trip-level Parquet panel. Each household–year–trip key must be unique under the default settings. Demographics and life events must be constant within a household-year. The read-only views merge milk into beverages, combine low- and medium-sugar cereal, and correct negative component amounts within the rounding tolerance.

The two likelihood blocks use different records. Every retained trip supplies five category purchase indicators for incidence. A category with positive included spending also supplies a composition observation. A positive purchase code with zero included dollars contributes only to incidence. Every retained household-year remains in its state sequence, including years with no composition observations. By default, the next observed year is one transition even if a calendar year is missing.

In [ ]:
def sql_name(name):
    return '"' + name.replace('"', '""') + '"'


def register_combined_beverage_view(con, path):
    """Merge milk into beverages and collapse cereal codes in a read-only SQL view.

    The Parquet file is never edited. The milk category is treated as ordinary
    milk; there is no UPC/flavor screening and no sweetened-milk relabeling.
    Original milk, beverage and cereal codes are validated first, and an invalid
    or missing code stops the run. Cereal codes 1 and 2 merge into the healthy
    group; 3 maps to high sugar. Nulls are errors, never non-purchases.
    Yogurt needs no recoding here: the V41 panel already carries the sugar-based
    yogurt_y (0 none, 1 any regular-sugar, 2 all high-sugar).
    """
    con.from_parquet(str(path)).create_view("raw_source")
    actual = {r[0] for r in con.execute("DESCRIBE raw_source").fetchall()}
    missing = {"milk_y", "beverage_y", "cereal_y"} - actual
    if missing:
        raise ValueError(f"Missing source outcome columns: {missing}")
    invalid = con.execute("""
        SELECT count(*) FROM raw_source
        WHERE milk_y IS NULL OR beverage_y IS NULL OR cereal_y IS NULL
           OR NOT isfinite(CAST(cereal_y AS DOUBLE)) OR cereal_y NOT IN (0,1,2,3)
           OR NOT isfinite(CAST(milk_y AS DOUBLE))
           OR NOT isfinite(CAST(beverage_y AS DOUBLE))
           OR milk_y NOT IN (0,1,2) OR beverage_y NOT IN (0,1,2)
    """).fetchone()[0]
    if invalid:
        raise ValueError(
            f"{invalid:,} invalid original milk/beverage/cereal outcome rows; fix upstream."
        )
    con.execute("""
        CREATE VIEW raw AS SELECT * REPLACE (
            CASE WHEN milk_y IN (1,2) OR beverage_y = 1 THEN 1
                 WHEN beverage_y = 2 THEN 2 ELSE 0 END AS beverage_y,
            CASE WHEN cereal_y IN (1,2) THEN 1
                 WHEN cereal_y=3 THEN 2 ELSE 0 END AS cereal_y
        ) FROM raw_source
    """)


def register_share_view(con):
    """Validate and construct shares without changing purchase-incidence codes."""
    spending = [v for cfg in CATEGORY_CONFIG.values() for v in cfg["spending"]]
    con.execute(
        "CREATE VIEW clean_spending AS SELECT * REPLACE ("
        + ",".join(
            f"greatest(CAST({sql_name(v)} AS DOUBLE),0.0) AS {sql_name(v)}"
            for v in spending
        )
        + ") FROM raw"
    )
    fields = []
    for c, groups in SHARE_COMPONENTS.items():
        fields.append(
            "("
            + "+".join(sql_name(v) for group in groups for v in group)
            + f") AS {c}_category_spend"
        )
        for k, group in enumerate(groups, 1):
            fields.append(
                "("
                + "+".join(sql_name(v) for v in group)
                + f") AS {c}_component_{k}_spend"
            )
    con.execute(
        "CREATE VIEW share_amounts AS SELECT *,"
        + ",".join(fields)
        + " FROM clean_spending"
    )
    inconsistent = " OR ".join(
        f"({c}_y=0 AND {c}_category_spend>0)" for c in CATEGORY_NAMES
    )
    if con.execute(
        "SELECT count(*) FROM share_amounts WHERE " + inconsistent
    ).fetchone()[0]:
        raise ValueError(
            "Positive category spending with a nonpurchase code; fix upstream."
        )
    fields = []
    for c, groups in SHARE_COMPONENTS.items():
        for k in range(1, len(groups) + 1):
            fields.append(
                f"CASE WHEN {c}_category_spend>0 THEN {c}_component_{k}_spend/{c}_category_spend ELSE 0.0 END AS {c}_share_{k}"
            )
    con.execute(
        "CREATE VIEW share_trips AS SELECT *,"
        + ",".join(fields)
        + " FROM share_amounts"
    )

### Annual statistics and input audits

For each household-year and category, the emission needs three counts (pure preferred, pure other and mixed), plus the sums of the two log shares for mixed purchases. The incidence table retains the joint counts by household and exact trip food spending. The remaining annual spending and share columns support audits and reporting.

In [ ]:
def audit_categories(con):
    """Calculate the category diagnostics in two scans of the eligible trips."""
    share_audit = []
    audit_fields, correction_fields = [], []
    labels = {c: [] for c in CATEGORY_NAMES}
    for c, cfg in CATEGORY_CONFIG.items():
        fields = [
            "count(*) AS n_trips",
            f"count(*) FILTER (WHERE {c}_y>0) AS purchase_trips",
            f"count(*) FILTER (WHERE {c}_category_spend>0) AS positive_spend_trips",
            f"count(*) FILTER (WHERE {c}_y>0 AND {c}_category_spend=0) AS zero_spend_purchase_trips",
            f"count(*) FILTER (WHERE {c}_category_spend>0 AND {c}_share_1=0) AS preferred_share_zero_trips",
            f"count(*) FILTER (WHERE {c}_share_1=1) AS preferred_share_one_trips",
            f"count(*) FILTER (WHERE {c}_share_1>0 AND {c}_share_1<1) AS preferred_share_interior_trips",
        ]
        nonzero = "+".join(
            f"CAST({c}_component_{k}_spend>0 AS INTEGER)"
            for k in range(1, len(SHARE_COMPONENTS[c]) + 1)
        )
        fields.append(f"count(*) FILTER (WHERE ({nonzero})>1) AS mixed_component_trips")
        # Audit only (never used in estimation): does the priority code agree with spending?
        # Code 1 with no preferred-component dollars can occur when the preferred item had a zero price.
        # A code above 1 with preferred-component dollars should not occur and points to an upstream coding problem.
        fields.append(
            f"count(*) FILTER (WHERE {c}_y=1 AND {c}_category_spend>0 AND {c}_component_1_spend=0) AS code1_without_preferred_spend_trips"
        )
        fields.append(
            f"count(*) FILTER (WHERE {c}_y>1 AND {c}_component_1_spend>0) AS code_above1_with_preferred_spend_trips"
        )
        for expression in fields:
            sql, label = expression.rsplit(" AS ", 1)
            audit_fields.append(f"{sql} AS {c}_{label}")
            labels[c].append(label)
        corrected = "+".join(
            f"CAST({sql_name(v)}<0 AS BIGINT)" for v in cfg["spending"]
        )
        correction_fields.append(f"sum({corrected}) AS {c}")
    values = (
        con.execute("SELECT " + ",".join(audit_fields) + " FROM trips").df().iloc[0]
    )
    corrections = con.execute(
        "SELECT "
        + ",".join(correction_fields)
        + " FROM raw SEMI JOIN eligible USING(hhid)"
    ).fetchone()
    for c, corrected_count in zip(CATEGORY_NAMES, corrections):
        record = {label: values[f"{c}_{label}"] for label in labels[c]}
        record["tiny_negative_components_corrected"] = corrected_count
        record["category"] = c
        share_audit.append(record)
    return pd.DataFrame(share_audit)


def aggregate_annual_panel(con, controls):
    # The household key is renamed household_code internally; exported files rename it back to hhid.
    aggs = [
        "CAST(hhid AS VARCHAR) AS household_code",
        "CAST(panel_year AS INTEGER) AS panel_year",
        "count(*) AS n_trips",
        "sum(CAST(food_spend AS DOUBLE)) AS food_total_spend_sum",
        "min(CAST(purchase_date AS DATE)) AS first_purchase_date",
        "max(CAST(purchase_date AS DATE)) AS last_purchase_date",
    ]
    aggs += [f"first(CAST({c} AS DOUBLE)) AS {c}" for c in controls]
    for c, cfg in CATEGORY_CONFIG.items():
        aggs += [
            f"sum(CAST({c}_y>0 AS BIGINT)) AS {c}_purchase_n",
            "sum("
            + "+".join(f"CAST({v} AS DOUBLE)" for v in cfg["spending"])
            + f") AS {c}_spend_sum",
        ]
        aggs += [
            f"sum(CAST({c}_y={k} AS BIGINT)) AS {c}_{v}_n"
            for k, v in cfg["outcomes"].items()
            if k
        ]
        aggs += [f"sum(CAST({c}_category_spend>0 AS BIGINT)) AS {c}_share_observed_n"]
        for k, v in cfg["outcomes"].items():
            if k:
                aggs += [
                    f"sum({c}_share_{k}) AS {c}_{v}_share_sum",
                    f"sum({c}_component_{k}_spend) AS {c}_{v}_component_spend_sum",
                ]
    # Exact sufficient statistics for the normalized trip-level likelihood.
    # Compute each mixed log share from both component amounts for stability.
    for c in CATEGORY_NAMES:
        h, u = (f"{c}_component_{j}_spend" for j in (1, 2))
        mix = f"{h}>0 AND {u}>0"
        for suffix, condition in [
            ("healthy", f"{h}>0 AND {u}=0"),
            ("unhealthy", f"{h}=0 AND {u}>0"),
            ("mixed", mix),
        ]:
            aggs.append(f"count(*) FILTER (WHERE {condition}) AS {c}_{suffix}")
        for suffix, v in [("log_y", h), ("log_u", u)]:
            aggs.append(
                f"sum(CASE WHEN {mix} THEN ln({v})-ln({c}_category_spend) ELSE 0 END) AS {c}_{suffix}"
            )
    hy = con.execute(
        "SELECT "
        + ",".join(aggs)
        + " FROM trips GROUP BY hhid,panel_year ORDER BY household_code,panel_year"
    ).df()
    if hy.empty:
        raise ValueError("No households pass the observed-year filter.")
    gaps = hy.groupby("household_code", sort=False).panel_year.diff() > 1
    if YEAR_GAP_POLICY not in ("observed_step", "error"):
        raise ValueError("Invalid YEAR_GAP_POLICY")
    if gaps.any() and YEAR_GAP_POLICY == "error":
        raise ValueError(
            f"{gaps.sum()} observed-year gaps; explicit handling required."
        )
    hy["chain_step"] = hy.groupby("household_code", sort=False).cumcount() + 1
    return hy


def write_incidence_counts(con, hy, scratch_dir):
    # Preserve the joint distribution of trip food spend and category purchases.
    # Sorted household indices match the bootstrap multiplicity vector exactly.
    hh_index = pd.DataFrame(
        {
            "household_code": hy.household_code.drop_duplicates().to_numpy(),
            "hh_index": np.arange(hy.household_code.nunique()),
        }
    )
    con.register("hh_map", hh_index)
    inc_path = scratch_dir / "incidence_counts.parquet"
    counts = [f"count(*) FILTER (WHERE {c}_y>0) AS {c}" for c in CATEGORY_NAMES]
    sql = (
        "SELECT hh_index, CAST(food_spend AS DOUBLE)/10 AS spend10, count(*) AS n, "
        + ",".join(counts)
        + " FROM trips JOIN hh_map ON CAST(hhid AS VARCHAR)=household_code"
        + " GROUP BY hh_index, spend10 ORDER BY hh_index, spend10"
    )
    con.sql(sql).write_parquet(str(inc_path), overwrite=True)
    return inc_path

### Input validation and preparation

In [ ]:
def prepare_parquet(path, scratch_dir):
    """Validate the trip panel and construct the two likelihood inputs."""
    with duckdb.connect() as con:
        con.execute(f"SET threads={HMM_NUM_THREADS}")
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        scratch_dir = Path(scratch_dir)
        scratch_dir.mkdir(parents=True, exist_ok=True)
        con.execute("SET temp_directory='" + str(scratch_dir).replace("'", "''") + "'")
        register_combined_beverage_view(con, path)
        controls = [*GATE_COVARIATE_COLS, *TRANSITION_EVENT_COLUMNS]
        spends = [s for cfg in CATEGORY_CONFIG.values() for s in cfg["spending"]]
        required = [
            "hhid",
            "panel_year",
            "trip_id",
            "purchase_date",
            "food_spend",
            *controls,
            *[c + "_y" for c in CATEGORY_NAMES],
            *spends,
        ]
        actual = {r[0] for r in con.execute("DESCRIBE raw").fetchall()}
        if set(required) - actual:
            raise ValueError(f"Missing columns: {set(required)-actual}")
        bad = [f"{sql_name(c)} IS NULL" for c in required]
        numeric = [
            "panel_year",
            "food_spend",
            *controls,
            *spends,
            *[c + "_y" for c in CATEGORY_NAMES],
        ]
        bad += [f"NOT isfinite(CAST({sql_name(c)} AS DOUBLE))" for c in numeric]
        bad += [
            "panel_year NOT BETWEEN 2008 AND 2017",
            "panel_year != floor(panel_year)",
            "TRY_CAST(purchase_date AS DATE) IS NULL",
        ]
        bad += [f"{c} NOT IN (0,1)" for c in [*TRANSITION_EVENT_COLUMNS, "K_1"]]
        bad += [
            f'{c}_y NOT IN ({",".join(map(str,cfg["outcomes"]))})'
            for c, cfg in CATEGORY_CONFIG.items()
        ]
        bad += [
            "family_size_numeric <= 0",
            "age <= 0",
            "age > 120",
            "income10k_imputed < 0",
        ]
        bad += [
            f"CAST({sql_name(c)} AS DOUBLE)<{-SPENDING_NEGATIVE_TOL}" for c in spends
        ]
        n, b = con.execute(
            "SELECT count(*), count(*) FILTER (WHERE " + " OR ".join(bad) + ") FROM raw"
        ).fetchone()
        if b:
            raise ValueError(
                f"{b:,} invalid rows; fix upstream rather than silently filtering."
            )
        if not n:
            raise ValueError("Empty input Parquet.")
        if CHECK_DUPLICATE_TRIPS:
            duplicate = con.execute(
                "SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id HAVING count(*)>1 LIMIT 1"
            ).fetchone()
            if duplicate:
                raise ValueError("Duplicate household-year-trip key.")
        checks = " OR ".join(f"min({c}) != max({c})" for c in controls)
        if con.execute(
            f"SELECT 1 FROM raw GROUP BY hhid,panel_year HAVING {checks} LIMIT 1"
        ).fetchone():
            raise ValueError("Demographics/events vary within a household-year.")
        con.execute(
            f"CREATE TEMP TABLE eligible AS SELECT hhid FROM raw GROUP BY hhid HAVING count(DISTINCT panel_year)>={MIN_OBSERVED_YEARS_PER_HOUSEHOLD}"
        )
        register_share_view(con)
        con.execute(
            "CREATE VIEW trips AS SELECT r.* FROM share_trips r SEMI JOIN eligible e USING(hhid)"
        )
        share_audit = audit_categories(con)
        # Only nine combinations: cheap audit of the original codes and merged codes.
        merge_audit = con.execute("""
            SELECT r.milk_y AS original_milk_y, r.beverage_y AS original_beverage_y,
                   CASE WHEN r.milk_y IN (1,2) OR r.beverage_y=1 THEN 1
                        WHEN r.beverage_y=2 THEN 2 ELSE 0 END AS merged_beverage_y,
                   count(*) AS n_trips
            FROM raw_source r SEMI JOIN eligible e USING(hhid)
            GROUP BY 1,2,3 ORDER BY 1,2
        """).df()
        hy = aggregate_annual_panel(con, controls)
        inc_path = write_incidence_counts(con, hy, scratch_dir)
    return hy, share_audit, merge_audit, inc_path

## 4. Likelihood and state probabilities

For positive category spending, let $y=H/(H+U)$, where $H$ and $U$ are the two component amounts. Conditional on category $c$, state $s$ and demographics $x$, the distribution has three parts.

| Observed purchase | Likelihood contribution |
|---|---|
| Pure other, $y=0$ | $(1-m_{cs})(1-h_{cs}(x))$ |
| Pure preferred, $y=1$ | $(1-m_{cs})h_{cs}(x)$ |
| Mixed, $0<y<1$ | $m_{cs}\, f_{\mathrm{Beta}}(y;\mu_{cs}\phi,(1-\mu_{cs})\phi)$ |

Here $m_{cs}$ is the probability of a mixed purchase and $h_{cs}(x)$ is the probability of a preferred purchase conditional on a pure basket. The pure-purchase logit has category-specific intercepts, state shifts and demographic slopes shared across states within category. The mixed-share mean $\mu_{cs}$ has a category-specific logit intercept and a common state shift. The precision $\phi$ is shared across categories and states. Thus the expected preferred share is $(1-m_{cs})h_{cs}(x)+m_{cs}\mu_{cs}$.

Both models estimate the initial state-2 probability. The HMM has a separate transition logit for each origin state and sums over possible paths with a forward–backward recursion. The LCM sums over two possible fixed household classes. Contributions multiply across trips and categories conditional on the state and covariates.

In [ ]:
_TINY = np.finfo(float).tiny


def _scaled_recursion(em, trans, initial):
    """Forward-backward with per-step scaling for households observed for T years.

    em: (2, T, n) log emissions by state; trans: (4, T, n) transition probabilities into
    year t in the order 1->1, 1->2, 2->1, 2->2 (t = 0 unused); initial: (2,) initial state
    probabilities. Returns household log likelihoods (n,), gamma (2, T, n) and xi
    (4, T, n) with xi[:, 0] = 0.
    """
    _, n_years, n = em.shape
    peak = np.maximum(em[0], em[1])
    b0 = np.maximum(np.exp(em[0] - peak), _TINY)  # emission likelihoods, rescaled
    b1 = np.maximum(np.exp(em[1] - peak), _TINY)
    p11, p12, p21, p22 = trans

    # Forward pass: filtered state probabilities a and their normalizing constants.
    a0, a1, scale = np.empty((3, n_years, n))
    x0, x1 = initial[0] * b0[0], initial[1] * b1[0]
    scale[0] = x0 + x1
    a0[0], a1[0] = x0 / scale[0], x1 / scale[0]
    for t in range(1, n_years):
        x0 = (a0[t - 1] * p11[t] + a1[t - 1] * p21[t]) * b0[t]
        x1 = (a0[t - 1] * p12[t] + a1[t - 1] * p22[t]) * b1[t]
        scale[t] = x0 + x1
        a0[t], a1[t] = x0 / scale[t], x1 / scale[t]
    household_ll = np.log(scale).sum(axis=0) + peak.sum(axis=0)

    # Backward pass: smoothed state (gamma) and transition (xi) probabilities.
    gamma = np.empty((2, n_years, n))
    xi = np.zeros((4, n_years, n))
    back0, back1 = np.ones(n), np.ones(n)
    gamma[0, -1], gamma[1, -1] = a0[-1], a1[-1]
    for t in range(n_years - 2, -1, -1):
        future0 = b0[t + 1] * back0 / scale[t + 1]
        future1 = b1[t + 1] * back1 / scale[t + 1]
        step11, step12 = p11[t + 1] * future0, p12[t + 1] * future1
        step21, step22 = p21[t + 1] * future0, p22[t + 1] * future1
        xi[0, t + 1], xi[1, t + 1] = a0[t] * step11, a0[t] * step12
        xi[2, t + 1], xi[3, t + 1] = a1[t] * step21, a1[t] * step22
        back0, back1 = step11 + step12, step21 + step22
        gamma[0, t], gamma[1, t] = a0[t] * back0, a1[t] * back1
    gamma /= gamma.sum(axis=0)
    xi[:, 1:] /= xi[:, 1:].sum(axis=0)
    return household_ll, gamma, xi


def _log_batch(em, log_trans, initial):
    """Stable exact log recursion, including transitions beyond exp underflow."""
    alpha = np.empty_like(em)
    backward = np.zeros_like(em)
    alpha[:, 0] = np.log(initial)[:, None] + em[:, 0]
    for t in range(1, em.shape[1]):
        for s in (0, 1):
            alpha[s, t] = em[s, t] + np.logaddexp(
                alpha[0, t - 1] + log_trans[0, s, t],
                alpha[1, t - 1] + log_trans[1, s, t],
            )
    ll = np.logaddexp(alpha[0, -1], alpha[1, -1])
    for t in range(em.shape[1] - 2, -1, -1):
        nxt = em[:, t + 1] + backward[:, t + 1]
        for r in (0, 1):
            backward[r, t] = np.logaddexp(
                log_trans[r, 0, t + 1] + nxt[0], log_trans[r, 1, t + 1] + nxt[1]
            )
    gamma = np.exp(alpha + backward - ll)
    gamma /= gamma.sum(axis=0)
    xi = np.zeros((4, *em.shape[1:]))
    for t in range(1, em.shape[1]):
        for r in (0, 1):
            for s in (0, 1):
                xi[2 * r + s, t] = np.exp(
                    alpha[r, t - 1]
                    + log_trans[r, s, t]
                    + em[s, t]
                    + backward[s, t]
                    - ll
                )
    xi[:, 1:] /= xi[:, 1:].sum(axis=0)
    return ll, gamma, xi


def _fast_forward_backward(arrays, emission, pi, trans):
    """Scaled batched recursion with exact log fallback before underflow occurs."""
    eta1, eta2 = (arrays.event_profiles @ trans.T).T
    log_trans = np.stack(
        [
            -np.logaddexp(0, eta1),
            -np.logaddexp(0, -eta1),
            -np.logaddexp(0, eta2),
            -np.logaddexp(0, -eta2),
        ]
    ).reshape(2, 2, -1)
    probabilities = np.exp(log_trans).reshape(4, -1)
    initial = np.array([1 - pi, pi])
    gamma = np.empty_like(emission)
    xi = np.zeros((len(emission), 4))
    household_ll = np.empty(len(arrays.hh_starts))
    for households, rows in arrays._fast_batches:
        profile = arrays.event_index[rows]
        em = emission.T[:, rows]
        # Only exceptional household paths require the slower log recursion.
        extreme = (np.abs(em[1] - em[0]) > 700).any(axis=0)
        if log_trans.min() < -700:
            extreme |= (log_trans[:, :, profile] < -700).any(axis=(0, 1, 2))
        for use_log in np.unique(extreme):
            group = (
                slice(None)
                if extreme.all() or not extreme.any()
                else (extreme == use_log)
            )
            group_em = em[:, :, group]
            group_profile = profile[:, group]
            if not use_log:
                with np.errstate(all="ignore"):
                    ll, g, x = _scaled_recursion(
                        group_em, probabilities[:, group_profile], initial
                    )
                use_log = not (
                    np.isfinite(ll).all()
                    and np.isfinite(g).all()
                    and np.isfinite(x).all()
                )
            if use_log:
                ll, g, x = _log_batch(group_em, log_trans[:, :, group_profile], initial)
            household_ll[households[group]] = ll
            gamma[rows[:, group]] = np.moveaxis(g, 0, -1)
            xi[rows[:, group]] = np.moveaxis(x, 0, -1)
    return float(arrays._hh_weights @ household_ll), gamma, xi, probabilities[[1, 3]]


def _fast_no_switch(arrays, emission, pi):
    household_emission = np.add.reduceat(emission, arrays.hh_starts, axis=0)
    joint = household_emission + np.array([np.log1p(-pi), np.log(pi)])
    ll = np.logaddexp(joint[:, 0], joint[:, 1])
    gamma = np.exp(joint - ll[:, None])
    gamma /= gamma.sum(axis=1, keepdims=True)
    return float(arrays._hh_weights @ ll), np.repeat(
        gamma, arrays.hh_ends - arrays.hh_starts, axis=0
    )

### Composition parameters and analytic score

Within each sample the two models share the annual arrays. State 2 is oriented toward the higher predicted preferred share, averaged equally over categories and across household-years with their bootstrap multiplicities. The HMM adds 14 transition coefficients to the LCM’s 48 composition parameters.

In [ ]:
class Composition:
    """Annual composition data and the likelihood for one household sample."""

    def __init__(self, hy, weights):
        self.hh_starts = np.flatnonzero(~hy.household_code.duplicated())
        self.hh_ends = np.r_[self.hh_starts[1:], len(hy)]
        lengths = self.hh_ends - self.hh_starts
        self._hh_weights = np.asarray(weights, float)
        self.w = np.repeat(self._hh_weights, lengths)
        self._fast_batches = [
            (
                np.flatnonzero(lengths == t),
                self.hh_starts[lengths == t][None, :] + np.arange(t)[:, None],
            )
            for t in np.unique(lengths)
        ]
        self.transition_x = np.column_stack(
            [np.ones(len(hy)), hy[TRANSITION_EVENT_COLUMNS]]
        )
        # At most 64 distinct combinations of the six binary life events.
        self.event_profiles, self.event_index = np.unique(
            self.transition_x, axis=0, return_inverse=True
        )
        self.event_index = self.event_index.ravel()
        self.X = np.column_stack(
            [
                np.ones(len(hy)),
                hy[GATE_COVARIATE_COLS].to_numpy() / np.array([1.0, 1.0, 1.0, 10.0]),
            ]
        )

        # Columns of the linear block are pure counts, mixed counts and sums
        # of the two log shares. Each category contributes one column per block.
        def annual_matrix(suffix):
            return hy[[f"{c}_{suffix}" for c in CATEGORY_NAMES]].to_numpy(dtype=float)

        self.healthy = annual_matrix("healthy")
        self.unhealthy = annual_matrix("unhealthy")
        self.pure = self.healthy + self.unhealthy
        mixed = annual_matrix("mixed")
        self.stats = np.ascontiguousarray(
            np.column_stack(
                [self.pure, mixed, annual_matrix("log_y"), annual_matrix("log_u")]
            )
        )
        pure_count, mixed_count = self.w @ self.pure, self.w @ mixed
        if np.any(np.minimum(pure_count, mixed_count) <= 0):
            raise ValueError(
                "At least one category lacks pure or mixed purchases in this sample."
            )
        self.initial_mix_logit = np.log(mixed_count / pure_count)

    def evaluate(self, theta, switching, details=False):
        """Return the negative mean household log likelihood and analytic score.

        With details=True, return the total log likelihood, posterior state
        probabilities and the category-level conditional means for reporting.
        """
        pure_coef = theta[:30].reshape(5, 6)
        mix_logit = theta[38:48].reshape(2, 5)
        mix = expit(mix_logit)
        phi, pi = np.exp(theta[36]), expit(theta[37])
        mean_logit = np.stack([theta[30:35], theta[30:35] + theta[35]])
        mu = expit(mean_logit)
        alpha, beta = phi * mu, phi * expit(-mean_logit)

        # Linear sufficient-statistic terms of both state emissions in one product.
        factors = np.column_stack(
            [
                -np.logaddexp(0, mix_logit),
                -np.logaddexp(0, -mix_logit) - betaln(alpha, beta),
                alpha - 1,
                beta - 1,
            ]
        )
        emission = self.stats @ factors.T
        base_logit = self.X @ pure_coef[:, :5].T
        pure_prob = []
        for state in range(2):
            z = base_logit if state == 0 else base_logit + pure_coef[:, 5]
            # Stable log Bernoulli probabilities. One softplus per observation
            # serves both outcomes and preserves accuracy near either endpoint.
            tail = np.log1p(np.exp(-np.abs(z)))
            emission[:, state] += (
                self.healthy * np.minimum(z, 0)
                - self.unhealthy * np.maximum(z, 0)
                - self.pure * tail
            ).sum(axis=1)
            pure_prob.append(expit(z))

        if switching:
            trans = theta[48:].reshape(2, 7)
            ll, posterior, xi, prob_state2 = _fast_forward_backward(
                self, emission, pi, trans
            )
        else:
            ll, posterior = _fast_no_switch(self, emission, pi)
        if details:
            shares = np.stack(
                [expit(-mix_logit[s]) * pure_prob[s] + mix[s] * mu[s] for s in (0, 1)],
                axis=2,
            )
            return ll, posterior, shares

        weights = self.w[:, None] * posterior
        pure_n, mixed_n, sum_log_y, sum_log_u = np.split(
            weights.T @ self.stats, 4, axis=1
        )
        grad = np.zeros_like(theta)
        grad[38:48] = (mixed_n - (mixed_n + pure_n) * mix).ravel()
        d_alpha = sum_log_y + mixed_n * (digamma(phi) - digamma(alpha))
        d_beta = sum_log_u + mixed_n * (digamma(phi) - digamma(beta))
        mean_score = (d_alpha - d_beta) * mu * beta
        grad[30:35] = mean_score.sum(axis=0)
        grad[35] = mean_score[1].sum()
        grad[36] = (d_alpha * alpha + d_beta * beta).sum()
        grad[37] = self._hh_weights @ (posterior[self.hh_starts, 1] - pi)
        pure_grad = grad[:30].reshape(5, 6)
        residual1 = weights[:, 1, None] * (self.healthy - self.pure * pure_prob[1])
        pure_grad[:, 5] = residual1.sum(axis=0)
        residual1 += weights[:, 0, None] * (self.healthy - self.pure * pure_prob[0])
        pure_grad[:, :5] = residual1.T @ self.X
        if switching:
            for state in (0, 1):
                pair = xi[:, 2 * state : 2 * state + 2]
                residual = (
                    pair[:, 1] - pair.sum(axis=1) * prob_state2[state, self.event_index]
                )
                grad[48 + 7 * state : 55 + 7 * state] = self.transition_x.T @ (
                    self.w * residual
                )
        n_households = self._hh_weights.sum()
        return -ll / n_households, -grad / n_households

    def orient(self, theta, switching):
        _, _, shares = self.evaluate(theta, switching, True)
        if (
            np.average((shares[:, :, 1] - shares[:, :, 0]).mean(axis=1), weights=self.w)
            >= 0
        ):
            return theta
        t = theta.copy()
        p = t[:30].reshape(5, 6)
        p[:, 0] += p[:, 5]
        p[:, 5] *= -1
        t[30:35] += t[35]
        t[35] *= -1
        t[37] *= -1
        t[38:48] = theta[38:48].reshape(2, 5)[::-1].ravel()
        if switching:
            t[48:] = -theta[48:].reshape(2, 7)[::-1].ravel()
        return t


def parameter_names(switching):
    names = [
        f"{c}_pure_{v}"
        for c in CATEGORY_NAMES
        for v in [
            "intercept",
            "K_1",
            "income10k",
            "family_size",
            "age_decades",
            "state2",
        ]
    ]
    names += [f"{c}_mixed_mean_intercept" for c in CATEGORY_NAMES]
    names += ["mixed_mean_state2", "log_precision", "initial_logit_state2"]
    names += [f"{c}_mixed_logit_state{s}" for s in (1, 2) for c in CATEGORY_NAMES]
    if switching:
        names += [f"trans_from{s}_{v}" for s in (1, 2) for v in TRANSITION_X_COLS]

    return names

### Optimization and state-independent incidence

The composition optimizer uses analytic gradients and L-BFGS-B. Each full-sample start has a fixed position in its random sequence, including when starts run concurrently. An accepted start must converge to a finite objective, have a maximum absolute gradient below 0.001 on the per-household objective, and finish away from the parameter bounds. The fit uses the highest likelihood among accepted starts.

In [ ]:
def initial_values(model, switching, rng, nstarts, warm=None):
    starts = []
    if warm is not None:
        starts.append(warm.copy())
    while len(starts) < nstarts:
        t = rng.normal(0, 0.18, len(parameter_names(switching)))
        t[:30].reshape(5, 6)[:, 5] = rng.normal(1, 0.6, 5)
        t[35:38] = [rng.uniform(0.2, 1.2), np.log(5), 0]
        t[38:48] += np.tile(model.initial_mix_logit, 2)
        if switching:
            t[[48, 55]] = [-1.5, 1.5]
        starts.append(t)
    return starts


def fit_one_start(indexed_start, model, switching):
    i, t = indexed_start
    # Numerical guard on logits/precision; flag boundary solutions for review.
    bounds = [(-30, 30)] * len(t)
    bounds[36] = (-10, 15)
    res = minimize(
        model.evaluate,
        t,
        args=(switching,),
        jac=True,
        method="L-BFGS-B",
        bounds=bounds,
        options=dict(maxiter=1000, ftol=1e-13, gtol=1e-6, maxls=50),
    )
    at_bound = any(min(x - lo, hi - x) < 1e-6 for x, (lo, hi) in zip(res.x, bounds))
    good = (
        res.success
        and np.isfinite(res.fun)
        and np.max(np.abs(res.jac)) < 0.001
        and not at_bound
    )

    diagnostics = dict(
        start=i,
        accepted=bool(good),
        boundary=bool(at_bound),
        message=str(res.message),
        max_gradient=float(np.max(np.abs(res.jac))),
        log_likelihood=float(-res.fun * model._hh_weights.sum()),
    )
    return res.x, diagnostics


def fit(model, switching, rng, nstarts, warm=None, workers=1):
    starts = initial_values(model, switching, rng, nstarts, warm)
    solve = partial(fit_one_start, model=model, switching=switching)
    if workers == 1:
        results = list(map(solve, enumerate(starts)))
    else:
        with ThreadPoolExecutor(max_workers=min(workers, nstarts)) as pool:
            results = list(pool.map(solve, enumerate(starts)))
    log = [row for _, row in results]
    accepted = [(theta, row) for theta, row in results if row["accepted"]]
    if not accepted:
        raise RuntimeError(
            f"No accepted optimum. Best attempt: {max(log,key=lambda r:r['log_likelihood'])}"
        )
    # pool.map retains start order, including the tie break.
    best, _ = max(accepted, key=lambda result: result[1]["log_likelihood"])
    return model.orient(best, switching), log

For trip food spending $F$ in dollars, the category purchase probability is $\operatorname{logit}^{-1}(b_{0c}+b_{1c}F/10)$. Each slope is a conditional log-odds difference per $10 of total food spending, which includes the category itself. The ten coefficients are shared by both models and estimated once per sample. Adding the incidence log likelihood to the composition log likelihood gives the joint likelihood used for AIC and BIC.

In [ ]:
def fit_incidence(path, weights, threads=1):
    """Exact grouped Bernoulli likelihood, conditional on total trip food spend."""
    bootstrap_weights = pd.DataFrame(
        {"hh_index": np.arange(len(weights)), "w": weights}
    )
    with TemporaryDirectory(
        dir=path.parent, prefix="inc_"
    ) as temp_dir, duckdb.connect() as con:
        con.execute("SET temp_directory='" + temp_dir.replace("'", "''") + "'")
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute(f"SET threads={threads}")
        con.register("hh_weights", bootstrap_weights)
        fields = [f"sum(d.{col}*w) AS {col}" for col in ["n", *CATEGORY_NAMES]]
        sql = (
            "SELECT spend10," + ",".join(fields) + " FROM read_parquet(?) d "
            "JOIN hh_weights USING(hh_index) WHERE w>0 GROUP BY spend10 ORDER BY spend10"
        )
        counts = con.execute(sql, [str(path)]).df()
    x = counts.spend10.to_numpy(float)
    total = counts.n.to_numpy(float)
    center = np.average(x, weights=total)
    scale = np.sqrt(np.average((x - center) ** 2, weights=total))
    if not scale > 0:
        raise ValueError("No variation in total trip food spending.")
    design = np.column_stack([np.ones(len(x)), (x - center) / scale])
    n_trips = total.sum()
    coefficients, ll = [], 0.0
    for c in CATEGORY_NAMES:
        bought = counts[c].to_numpy(float)
        missed = total - bought
        if min(bought.sum(), missed.sum()) <= 0:
            raise ValueError(f"No incidence variation for {c}.")

        def objective(beta):
            eta = design @ beta
            loss = bought @ np.logaddexp(0, -eta) + missed @ np.logaddexp(0, eta)
            grad = design.T @ (total * expit(eta) - bought)
            return loss / n_trips, grad / n_trips

        res = minimize(
            objective,
            [logit(bought.sum() / n_trips), 0.0],
            jac=True,
            method="L-BFGS-B",
            options={"ftol": 1e-14, "gtol": 1e-10},
        )
        if not res.success or np.max(np.abs(res.jac)) > 1e-6:
            raise RuntimeError(f"Incidence fit failed for {c}: {res.message}")
        slope = res.x[1] / scale
        coefficients.extend([res.x[0] - slope * center, slope])
        ll -= res.fun * n_trips
    return np.asarray(coefficients), ll

## 5. Spending predictions, model comparisons and inference

The spending export multiplies each observed household-year category budget by the posterior-weighted expected preferred share. This in-sample allocation assumes that the conditional mean share applies at the observed budget. Each row also retains the observed preferred dollars and both state-specific shares.

The joint parameter counts are **58 for the LCM** and **72 for the HMM**. Both models use the same sample. BIC uses the number of households.

In [ ]:
def estimates_with_mixing(theta):
    # Differences are computed within each aligned household bootstrap replicate.
    m1, m2 = expit(theta[38:48].reshape(2, 5))
    return np.r_[theta, m1, m2, m2 - m1]


def export_fit(model, hy, theta, name, out):
    switching = name == "hmm"
    ll, g, shares = model.evaluate(theta, switching, True)
    keys = hy[["household_code", "panel_year"]].rename(
        columns={"household_code": "hhid"}
    )
    posterior = keys.copy()
    posterior[["prob_state1", "prob_state2"]] = g
    posterior.to_csv(out / f"{name}_posterior.csv", index=False)
    prediction_path = out / f"{name}_conditional_spending.csv"
    for j, c in enumerate(CATEGORY_NAMES):
        d = keys.copy()
        d["category"] = c
        d["observed_budget"] = hy[f"{c}_spend_sum"]
        d["observed_healthy_spend"] = hy[
            f'{c}_{CATEGORY_CONFIG[c]["outcomes"][1]}_component_spend_sum'
        ]
        d[["healthy_share_state1", "healthy_share_state2"]] = shares[:, j, :]
        d["fitted_share"] = (shares[:, j, :] * g).sum(axis=1)
        d["fitted_healthy_spend"] = d.observed_budget * d.fitted_share
        d["fitted_unhealthy_spend"] = d.observed_budget - d.fitted_healthy_spend
        d.to_csv(
            prediction_path, mode="w" if j == 0 else "a", header=(j == 0), index=False
        )
    if switching:
        transitions = keys.copy()
        transitions[["p12", "p22"]] = expit(
            model.transition_x @ theta[48:].reshape(2, 7).T
        )
        transitions = transitions.drop(index=model.hh_starts)
        transitions.to_csv(out / "hmm_transition_probabilities.csv", index=False)
    k = len(theta)
    nh = len(model.hh_starts)
    return dict(
        model=name,
        log_likelihood=ll,
        k=k,
        households=nh,
        AIC=-2 * ll + 2 * k,
        BIC=-2 * ll + k * np.log(nh),
    )

### Paired household bootstrap

A draw resamples complete household histories with replacement. The LCM and HMM use the same household multiplicities and incidence fit. Each composition model has two starts per draw, one at the aligned full-sample estimate and one randomized. The state-specific mixed-purchase probabilities and their differences are recomputed after state alignment in every accepted draw.

The parameter tables report accepted draws by model. Covariance matrices, bootstrap standard errors, percentile intervals and normal-reference p-values are produced when at least 80% of draws are accepted, with a minimum of two. With the default 500 draws, the threshold is 400 per model.

In [ ]:
def save_checkpoint(path, record):
    """Publish a complete result after it has been serialized successfully."""
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(record), encoding="utf-8")
    temporary.replace(path)


def checkpoint_directory(out, source, nstarts):
    """Identify the precise source file, statistical specification and software."""
    with source.open("rb") as stream:
        key = hashlib.sha256()
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            key.update(block)
    definitions = (
        prepare_parquet,
        register_combined_beverage_view,
        register_share_view,
        sql_name,
        aggregate_annual_panel,
        write_incidence_counts,
        Composition.__init__,
        Composition.evaluate,
        Composition.orient,
        parameter_names,
        initial_values,
        fit,
        fit_one_start,
        fit_incidence,
        _fast_forward_backward,
        _fast_no_switch,
        _scaled_recursion,
        _log_batch,
        estimates_with_mixing,
        bootstrap_draw,
        run_models,
    )
    for definition in definitions:
        key.update(inspect.getsource(definition).encode())
    settings = [
        SEED,
        nstarts,
        MIN_OBSERVED_YEARS_PER_HOUSEHOLD,
        YEAR_GAP_POLICY,
        CHECK_DUPLICATE_TRIPS,
        SPENDING_NEGATIVE_TOL,
        CATEGORY_NAMES,
        CATEGORY_CONFIG,
        SHARE_COMPONENTS,
        TRANSITION_EVENT_COLUMNS,
        GATE_COVARIATE_COLS,
        np.__version__,
        scipy.__version__,
        pd.__version__,
        duckdb.__version__,
    ]
    key.update(json.dumps(settings).encode())
    folder = out / ("bootstrap_" + key.hexdigest()[:20])
    folder.mkdir(exist_ok=True)
    return folder

In [ ]:
def bootstrap_draw(b, annual, household_index, inc_path, fits, folder):
    """Refit incidence, LCM and HMM on the same resampled household histories."""
    nh = household_index[-1] + 1
    rng = np.random.default_rng([SEED, b, 44])
    weights = np.bincount(rng.integers(nh, size=nh), minlength=nh)
    record = {}
    try:
        boot_inc, _ = fit_incidence(inc_path, weights)
        # A household drawn k times contributes k times its full path likelihood.
        # Omitted households need neither emissions nor a forward/backward pass.
        selected = weights > 0
        model = Composition(annual.loc[selected[household_index]], weights[selected])
    except (ValueError, RuntimeError) as error:
        record = {name: str(error) for name in fits}
    else:
        for name, theta in fits.items():
            try:
                t, _ = fit(model, name == "hmm", rng, 2, warm=theta)
                record[name] = np.r_[estimates_with_mixing(t), boot_inc].tolist()
            except (ValueError, RuntimeError) as error:
                record[name] = str(error)
    save_checkpoint(folder / f"draw_{b:04d}.json", record)
    return b, record


def run_bootstrap(hy, inc_path, fits, folder, reps):
    pending = [b for b in range(reps) if not (folder / f"draw_{b:04d}.json").exists()]
    if pending:
        # Both models use the same annual arrays; exclude audit/export columns
        # before building each worker's resampled panel.
        columns = ["household_code", *GATE_COVARIATE_COLS, *TRANSITION_EVENT_COLUMNS]
        columns += [
            f"{c}_{v}"
            for c in CATEGORY_NAMES
            for v in ("healthy", "unhealthy", "mixed", "log_y", "log_u")
        ]
        annual = hy[columns]
        household_index = np.cumsum(~annual.household_code.duplicated().to_numpy()) - 1
        solve = partial(
            bootstrap_draw,
            annual=annual,
            household_index=household_index,
            inc_path=inc_path,
            fits=fits,
            folder=folder,
        )
        finished = reps - len(pending)

        def report(result):
            nonlocal finished
            b, record = result
            finished += 1
            status = ", ".join(
                f"{name}={isinstance(values,list)}" for name, values in record.items()
            )
            print(
                f"Bootstrap draw {b+1} ({finished}/{reps} complete): {status}",
                flush=True,
            )

        if FIT_WORKERS == 1:
            for b in pending:
                report(solve(b))
        else:
            with ThreadPoolExecutor(max_workers=min(FIT_WORKERS, len(pending))) as pool:
                futures = [pool.submit(solve, b) for b in pending]
                for future in as_completed(futures):
                    report(future.result())
    return [
        json.loads((folder / f"draw_{b:04d}.json").read_text(encoding="utf-8"))
        for b in range(reps)
    ]


def summarize_bootstrap(out, draws, point_tables):
    for name, tab in point_tables.items():
        valid = [r[name] for r in draws if isinstance(r[name], list)]
        tab["accepted_draws"] = len(valid)
        covariance_path = out / f"{name}_bootstrap_covariance.csv"
        if len(valid) >= max(2, np.ceil(0.8 * len(draws))):
            samples = pd.DataFrame(valid, columns=tab.parameter.tolist())
            samples.cov().to_csv(covariance_path)
            tab["bootstrap_se"] = samples.std().values
            tab[["lower_95", "upper_95"]] = samples.quantile([0.025, 0.975]).T.values
            tab["z"] = tab.estimate / tab.bootstrap_se.replace(0, np.nan)
            tab["p_value"] = 2 * norm.sf(abs(tab.z))
        else:
            covariance_path.unlink(missing_ok=True)
            print(
                f"{name}: insufficient accepted draws for inference ({len(valid)}/{len(draws)})."
            )
        tab.to_csv(out / f"{name}_parameters.csv", index=False)

### Full-sample fits and joint model comparison

In [ ]:
def run_models(hy, out, synthetic, inc_path, source):
    nh = hy.household_code.nunique()
    weights = np.ones(nh)
    nstarts = 3 if synthetic else N_STARTS
    reps = 3 if synthetic else BOOTSTRAP_REPS
    folder = checkpoint_directory(out, source, nstarts)
    print(
        f"{nh:,} households | {nstarts} starts per model | {reps} bootstrap draws | {FIT_WORKERS} worker(s)",
        flush=True,
    )
    model = Composition(hy, weights)
    inc_names = [
        f"{c}_inc_{v}"
        for c in CATEGORY_NAMES
        for v in ("intercept", "food_total_spend_10")
    ]
    mix_names = [
        f"{c}_mix_{v}"
        for v in ("prob_state1", "prob_state2", "difference_state2_minus_state1")
        for c in CATEGORY_NAMES
    ]

    incidence_file = folder / "point_incidence.json"
    if incidence_file.exists():
        inc, inc_ll = json.loads(incidence_file.read_text(encoding="utf-8"))
        inc = np.asarray(inc)
    else:
        inc, inc_ll = fit_incidence(inc_path, weights, threads=HMM_NUM_THREADS)
        save_checkpoint(incidence_file, [inc.tolist(), inc_ll])

    fits, point_tables, comparison = {}, {}, []
    for idx, name in enumerate(("lcm", "hmm")):
        switching = name == "hmm"
        point_file = folder / f"point_{name}.json"
        if point_file.exists():
            point = json.loads(point_file.read_text(encoding="utf-8"))
            theta, log = np.asarray(point["theta"]), point["log"]
            print(f"Reusing {name.upper()} point fit.", flush=True)
        else:
            theta, log = fit(
                model,
                switching,
                np.random.default_rng(SEED + idx),
                nstarts,
                workers=FIT_WORKERS,
            )
            save_checkpoint(point_file, {"theta": theta.tolist(), "log": log})
        fits[name] = theta
        pd.DataFrame(log).to_csv(out / f"{name}_multistart.csv", index=False)
        comparison.append(export_fit(model, hy, theta, name, out))
        names = parameter_names(switching) + mix_names + inc_names
        point_tables[name] = pd.DataFrame(
            {"parameter": names, "estimate": np.r_[estimates_with_mixing(theta), inc]}
        )
        point_tables[name].to_csv(out / f"{name}_parameters.csv", index=False)

    comp = pd.DataFrame(comparison)
    comp["composition_log_likelihood"] = comp.log_likelihood
    comp["incidence_log_likelihood"] = inc_ll
    comp["composition_k"] = comp.k
    comp["k"] += len(inc)
    comp["log_likelihood"] += inc_ll
    comp["AIC"] = -2 * comp.log_likelihood + 2 * comp.k
    comp["BIC"] = -2 * comp.log_likelihood + comp.k * np.log(nh)
    for col in ("AIC", "BIC"):
        comp["delta_" + col] = comp[col] - comp[col].min()
    comp.to_csv(out / "model_comparison.csv", index=False)
    print(comp.to_string(index=False))
    del model
    draws = run_bootstrap(hy, inc_path, fits, folder, reps)
    summarize_bootstrap(out, draws, point_tables)

## 6. Synthetic fixture

The synthetic run generates a schema-compatible Parquet file in its own results folder. It provides a small end-to-end check of both estimators and all exports.

In [ ]:
def make_synthetic_parquet(path, n_households=120, seed=SEED):
    """Artificial fixture with all schema fields and both states; never research data."""
    rng = np.random.default_rng(seed)
    rows = []
    trip = 0
    for h in range(n_households):
        state = int(rng.random() < 0.3)
        for t in range(4):
            events = rng.binomial(1, 0.15, 6)
            if t:
                state = int(rng.random() < (0.8 if state else 0.2))
            controls = [
                rng.binomial(1, 0.5),
                rng.uniform(1, 12),
                rng.integers(1, 6),
                rng.uniform(25, 75),
            ]
            for j in range(12):
                trip += 1
                row = dict(
                    hhid=f"{h:06d}",
                    panel_year=2008 + t,
                    trip_id=str(trip),
                    purchase_date=pd.Timestamp(2008 + t, 1, 1)
                    + pd.Timedelta(days=j * 20),
                    food_spend=float(rng.integers(10, 100)),
                    initial_fresh_fv_share_2007=0.3,
                    other_unemployment=0,
                )
                row.update(zip(TRANSITION_EVENT_COLUMNS, events))
                row.update(zip(GATE_COVARIATE_COLS, controls))
                for c, cfg in CATEGORY_CONFIG.items():
                    for v in cfg["spending"]:
                        row[v] = 0.0
                    row[c + "_y"] = 0
                    if rng.random() < 0.7:
                        groups = (
                            SHARE_COMPONENTS[c]
                            if c != "beverage"
                            else [
                                ["healthy_beverage_spending"],
                                ["unhealthy_beverage_spending"],
                            ]
                        )
                        m = len(groups)
                        p = 0.85 if state else 0.15
                        probs = np.r_[p, np.repeat((1 - p) / (m - 1), m - 1)]
                        if rng.random() < 0.35:
                            q = np.eye(m)[rng.choice(m, p=probs)]
                        else:
                            q = rng.dirichlet(8 * probs)
                        total = rng.uniform(0.1, 15)
                        for group, share in zip(groups, q):
                            row[group[0]] = float(total * share)
                        row[c + "_y"] = int(np.flatnonzero(q > 0)[0] + 1)
                        if c == "cereal":
                            # Write the ORIGINAL three-level source schema; the read-only view merges it.
                            low = float(row["low_sugar_cereal_spending"]) * float(
                                rng.random()
                            )
                            row["medium_sugar_cereal_spending"] = (
                                row["low_sugar_cereal_spending"] - low
                            )
                            row["low_sugar_cereal_spending"] = low
                            row["cereal_y"] = (
                                1
                                if low > 0
                                else (
                                    2 if row["medium_sugar_cereal_spending"] > 0 else 3
                                )
                            )
                row["milk_y"] = int(rng.choice([0, 1, 2], p=[0.65, 0.15, 0.20]))
                milk_columns = [
                    "skimmed_milk_spending",
                    "low_fat_milk_spending",
                    "other_milk_spending",
                ]
                if row["milk_y"]:
                    column = (
                        milk_columns[0]
                        if row["milk_y"] == 1
                        else milk_columns[int(rng.integers(1, 3))]
                    )
                    row[column] = float(rng.uniform(0.1, 10))
                row["food_spend"] = sum(
                    row[v] for cfg in CATEGORY_CONFIG.values() for v in cfg["spending"]
                ) + float(rng.uniform(1, 30))
                rows.append(row)
    frame = pd.DataFrame(rows)
    with duckdb.connect() as con:
        con.from_df(frame).write_parquet(str(path))

## 7. Run the selected analysis

After estimation, open `model_comparison.csv` for the model comparison or a conditional-spending file for the household-year allocations.

| Output | Unit and contents |
|---|---|
| `model_comparison.csv` | One row per model with joint and component log likelihoods, parameter counts, AIC, BIC and differences from the best criterion |
| `{lcm,hmm}_parameters.csv`, `{lcm,hmm}_bootstrap_covariance.csv` | Coefficients, derived mixing probabilities and differences, plus household-bootstrap inference |
| `{lcm,hmm}_posterior.csv` | One row per household-year with both state probabilities |
| `{lcm,hmm}_conditional_spending.csv` | One row per household-year and category with observed budget, observed preferred dollars, fitted shares and fitted dollars |
| `hmm_transition_probabilities.csv` | Probabilities of entering state 2 from each origin state, for observed transitions |
| `{lcm,hmm}_multistart.csv` | One row per full-sample start with acceptance, optimizer status, gradient and likelihood |
| `annual_sufficient_statistics.csv`, `share_input_audit.csv`, `beverage_merge_audit.csv` | Annual panel, category spending diagnostics and source-to-merged beverage codes |

In [ ]:
if RUN_MODE not in ("synthetic", "validate_only", "full_estimation"):
    raise ValueError("Unknown RUN_MODE")
synthetic = RUN_MODE == "synthetic"
out = Path.cwd() / "V46_synthetic_results" if synthetic else OUTPUT_DIR
out.mkdir(exist_ok=True, parents=True)
source = INPUT_PATH
if synthetic:
    source = out / "synthetic.parquet"
    make_synthetic_parquet(source, n_households=240)
if not source.is_file():
    raise FileNotFoundError(source)
hy, audit, merge, inc_path = prepare_parquet(source, out / "duckdb_temp")
audit.to_csv(out / "share_input_audit.csv", index=False)
merge.to_csv(out / "beverage_merge_audit.csv", index=False)
hy.to_csv(out / "annual_sufficient_statistics.csv", index=False)
print(audit.to_string(index=False))
if RUN_MODE != "validate_only":
    limit = (
        threadpool_limits(limits=BLAS_THREADS, user_api="blas")
        if threadpool_limits
        else nullcontext()
    )
    with limit:
        run_models(hy, out, synthetic, inc_path, source)